# 🧠 Satria Data 2025 — Multimodal Emotion Classification

**Dataset:** MELD (Multimodal EmotionLines Dataset)  
**Task:** Emotion Recognition from Text + Audio + Video  
**Emotions:** Anger, Disgust, Fear, Joy, Neutral, Sadness, Surprise (7 classes)  
**Architecture:** DeBERTa-v3 (Text) + WavLM (Audio) + EfficientNet-B2 (Video) → Attention Fusion  

---

### How It Works
1. **Phase 1 — Feature Extraction:** Extract embeddings from all 3 modalities and save to disk  
2. **Phase 2 — Fusion Training:** Train a lightweight attention-based fusion model on the saved embeddings  

This two-phase approach keeps GPU memory usage low and allows re-training the fusion model without re-extracting features.

## Section 1: Setup & Configuration

In [ ]:

# === 1.1 Import Libraries ===

import os
import re
import gc
import glob
import string
import warnings
import random
import subprocess
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Audio
try:
    import torchaudio
    HAS_TORCHAUDIO = True
except ImportError:
    HAS_TORCHAUDIO = False
    print('⚠️ torchaudio not available, will use subprocess + ffmpeg fallback')

# Vision
import cv2
from torchvision import models, transforms

# NLP
from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification,
    AutoFeatureExtractor, WavLMModel
)

# Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.preprocessing import LabelEncoder

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print('✅ All libraries imported!')

In [ ]:
# === 1.2 Global Configuration ===

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# Matplotlib & Seaborn style
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams.update({'figure.figsize': (12, 6), 'figure.dpi': 100, 'font.size': 12})

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Emotion labels
EMOTION_LABELS = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']
NUM_CLASSES = len(EMOTION_LABELS)

# Embedding dimensions (from pre-trained models)
TEXT_DIM = 768     # DeBERTa-v3-base hidden size
AUDIO_DIM = 768    # WavLM-base hidden size
VIDEO_DIM = 1408   # EfficientNet-B2 feature dim (after removing final FC)

# Feature save paths
FEATURE_DIR = '/kaggle/working/features'
os.makedirs(FEATURE_DIR, exist_ok=True)

print(f'Device : {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
if DEVICE.type == 'cuda' and torch.cuda.device_count() > 1:
    print(f'GPUs   : {torch.cuda.device_count()}')
print(f'Seed   : {RANDOM_SEED}')
print(f'PyTorch: {torch.__version__}')
print('✅ Configuration complete!')

## Section 2: Load MELD Dataset & Map Video Paths

In [ ]:
# === 2.1 Load MELD CSVs ===

# Auto-detect MELD-RAW path
MELD_RAW = None
for candidate in [
    '/kaggle/input/meld-dataset/MELD-RAW/MELD.Raw',
    '/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw',
    '/kaggle/input/meld-dataset/MELD.Raw',
]:
    if os.path.isdir(candidate):
        MELD_RAW = candidate
        break

if MELD_RAW is None:
    # Fallback: search for the train CSV
    found = glob.glob('/kaggle/input/**/train_sent_emo.csv', recursive=True)
    if found:
        MELD_RAW = str(Path(found[0]).parent.parent)
        print(f'Auto-detected MELD_RAW: {MELD_RAW}')
    else:
        raise FileNotFoundError('Cannot find MELD dataset. Please attach the MELD dataset to this notebook.')

print(f'MELD_RAW path: {MELD_RAW}')

# Define CSV paths
csv_paths = {
    'train': os.path.join(MELD_RAW, 'train', 'train_sent_emo.csv'),
    'dev':   os.path.join(MELD_RAW, 'dev_sent_emo.csv'),
    'test':  os.path.join(MELD_RAW, 'test_sent_emo.csv'),
}

# Fallback if train CSV is at root level
if not os.path.isfile(csv_paths['train']):
    alt = os.path.join(MELD_RAW, 'train_sent_emo.csv')
    if os.path.isfile(alt):
        csv_paths['train'] = alt

# Load CSVs
datasets = {}
for split, path in csv_paths.items():
    if os.path.isfile(path):
        datasets[split] = pd.read_csv(path)
        print(f'✅ {split:5s} loaded: {datasets[split].shape}  <- {os.path.basename(path)}')
    else:
        print(f'⚠️ {split:5s} NOT FOUND: {path}')

df_train = datasets.get('train')
df_dev   = datasets.get('dev')
df_test  = datasets.get('test')

if df_train is not None:
    print(f'\nColumns: {df_train.columns.tolist()}')
    print(f'\nEmotion distribution (train):')
    print(df_train['Emotion'].value_counts().to_string())

In [ ]:
# === 2.2 Map Video File Paths ===
# Each utterance has a corresponding .mp4 file named like: dia0_utt0.mp4

def find_video_dirs():
    """Find directories containing .mp4 files for each split."""
    video_dirs = {}
    
    # Common MELD video directory patterns
    candidates = {
        'train': [
            os.path.join(MELD_RAW, 'train', 'train_splits'),
            os.path.join(MELD_RAW, 'train'),
            os.path.join(MELD_RAW, 'train_splits'),
        ],
        'dev': [
            os.path.join(MELD_RAW, 'dev', 'dev_splits_complete'),
            os.path.join(MELD_RAW, 'dev'),
            os.path.join(MELD_RAW, 'dev_splits_complete'),
        ],
        'test': [
            os.path.join(MELD_RAW, 'test', 'output_repeated_splits_test'),
            os.path.join(MELD_RAW, 'test'),
            os.path.join(MELD_RAW, 'output_repeated_splits_test'),
        ]
    }
    
    for split, paths in candidates.items():
        for p in paths:
            if os.path.isdir(p):
                mp4s = glob.glob(os.path.join(p, '*.mp4'))
                if mp4s:
                    video_dirs[split] = p
                    print(f'✅ {split:5s} videos: {len(mp4s)} files in {p}')
                    break
        if split not in video_dirs:
            # Brute force search
            found = glob.glob(os.path.join(MELD_RAW, '**', '*.mp4'), recursive=True)
            if found:
                # Group by directory
                from collections import defaultdict
                dir_counts = defaultdict(int)
                for f in found:
                    dir_counts[os.path.dirname(f)] += 1
                # Pick the directory with the most mp4s
                best_dir = max(dir_counts, key=dir_counts.get)
                video_dirs[split] = best_dir
                print(f'⚠️ {split:5s} videos: fallback to {best_dir} ({dir_counts[best_dir]} files)')
    
    return video_dirs

video_dirs = find_video_dirs()

def get_video_path(row, split):
    """Construct the expected video file path for a given utterance row."""
    if split not in video_dirs:
        return None
    dia_id = row.get('Dialogue_ID', 0)
    utt_id = row.get('Utterance_ID', 0)
    filename = f'dia{dia_id}_utt{utt_id}.mp4'
    path = os.path.join(video_dirs[split], filename)
    return path if os.path.isfile(path) else None

# Add video_path column to each dataframe
for split_name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
    if df is not None:
        df['video_path'] = df.apply(lambda r: get_video_path(r, split_name), axis=1)
        found = df['video_path'].notna().sum()
        total = len(df)
        print(f'{split_name}: {found}/{total} videos mapped ({found/total*100:.1f}%)')

## Section 3: Text Preprocessing

In [ ]:
# === 3.1 Text Cleaning & Dialogue Context ===
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Setup NLTK (offline-compatible)
kaggle_nltk_paths = ['/usr/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/share/nltk_data']
for p in kaggle_nltk_paths:
    if os.path.isdir(p) and p not in nltk.data.path:
        nltk.data.path.insert(0, p)

for pkg in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    try:
        nltk.data.find(f'tokenizers/{pkg}' if 'punkt' in pkg else f'corpora/{pkg}')
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

try:
    STOP_WORDS = set(stopwords.words('english'))
except Exception:
    STOP_WORDS = {'i','me','my','we','our','you','your','he','him','his','she','her','it','its','they','them','their','what','which','who','this','that','am','is','are','was','were','be','been','have','has','had','do','does','did','a','an','the','and','but','if','or','as','of','at','by','for','with','to','from','in','out','on','off','up','down'}

negation_words = {'no','not','nor','neither','never','none','but','against',"don't","can't","won't","shouldn't","couldn't","didn't","doesn't","haven't","hasn't","hadn't","isn't","aren't","wasn't","weren't"}
STOP_WORDS = STOP_WORDS - negation_words

CONTRACTIONS_MAP = {
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is",
    "it's": "it is", "we're": "we are", "they're": "they are", "i've": "i have",
    "don't": "do not", "doesn't": "does not", "didn't": "did not",
    "can't": "cannot", "won't": "will not", "wouldn't": "would not",
    "shouldn't": "should not", "couldn't": "could not"
}

lemmatizer = WordNetLemmatizer()
try:
    nltk.data.find('corpora/wordnet')
    USE_LEMMATIZER = True
except LookupError:
    USE_LEMMATIZER = False

def clean_text(text):
    if not isinstance(text, str): return ''
    text = text.lower()
    for c, e in CONTRACTIONS_MAP.items(): text = text.replace(c, e)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s!?]', ' ', text)
    text = re.sub(r'([!?])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = text.split()
    if USE_LEMMATIZER:
        tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 1 or t in ['!', '?']]
    else:
        tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 1 or t in ['!', '?']]
    return ' '.join(tokens)

# Apply cleaning + dialogue context
for df in [df_train, df_dev, df_test]:
    if df is not None:
        df['clean_text'] = df['Utterance'].apply(clean_text)

for df in [df_train, df_dev, df_test]:
    if df is not None:
        df.sort_values(['Dialogue_ID', 'Utterance_ID'], inplace=True)
        df.reset_index(drop=True, inplace=True)
        df['prev_clean_text'] = df.groupby('Dialogue_ID')['clean_text'].shift(1).fillna('')
        df['context_text'] = df.apply(
            lambda r: r['clean_text'] if r['prev_clean_text'] == '' else r['prev_clean_text'] + ' [SEP] ' + r['clean_text'],
            axis=1
        )

# Encode labels
le = LabelEncoder()
le.fit(EMOTION_LABELS)
y_train = le.transform(df_train['Emotion'])
y_dev   = le.transform(df_dev['Emotion'])
y_test  = le.transform(df_test['Emotion'])

print(f'Train: {len(df_train)} | Dev: {len(df_dev)} | Test: {len(df_test)}')
print('✅ Text preprocessing complete!')

---
## Phase 1: Feature Extraction

We extract embeddings from all 3 modalities and save them to disk.  
This is a **one-time cost** (~30-60 min). After this, you can retrain the fusion model in seconds.

In [ ]:
# === 4.1 Text Feature Extraction (DeBERTa-v3) ===

# Auto-detect model path
TEXT_MODEL_PATH = None
for candidate in [
    '/kaggle/input/models/alexxxsem/deberta-v3/pytorch/base/2',
    '/kaggle/input/deberta-v3-base',
    '/kaggle/input/deberta-v3/pytorch/base',
]:
    if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, 'config.json')):
        TEXT_MODEL_PATH = candidate
        break

if TEXT_MODEL_PATH is None:
    configs = glob.glob('/kaggle/input/**/config.json', recursive=True)
    for c in configs:
        with open(c) as f:
            if 'deberta' in f.read().lower():
                TEXT_MODEL_PATH = os.path.dirname(c)
                break

if TEXT_MODEL_PATH is None:
    raise FileNotFoundError('Cannot find DeBERTa model. Please attach the DeBERTa-v3 PyTorch model to this notebook.')

print(f'Text model: {TEXT_MODEL_PATH}')

# Check if features already extracted
text_feat_path = os.path.join(FEATURE_DIR, 'text_features.pt')
if os.path.isfile(text_feat_path):
    print(f'✅ Text features already extracted, loading from {text_feat_path}')
    text_features = torch.load(text_feat_path)
else:
    print('Extracting text features with DeBERTa-v3...')
    
    text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
    text_model = AutoModel.from_pretrained(TEXT_MODEL_PATH)
    text_model = text_model.to(DEVICE).float()
    text_model.eval()
    
    MAX_LEN = 64
    
    def extract_text_embeddings(texts, batch_size=64):
        """Extract [CLS] token embeddings from DeBERTa."""
        all_embeddings = []
        for i in tqdm(range(0, len(texts), batch_size), desc='Text features'):
            batch_texts = texts[i:i+batch_size]
            inputs = text_tokenizer(
                batch_texts, max_length=MAX_LEN,
                padding='max_length', truncation=True,
                return_tensors='pt'
            )
            input_ids = inputs['input_ids'].to(DEVICE)
            attention_mask = inputs['attention_mask'].to(DEVICE)
            
            with torch.no_grad():
                outputs = text_model(input_ids=input_ids, attention_mask=attention_mask)
                # Use [CLS] token embedding (first token)
                cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu()
            
            all_embeddings.append(cls_embeddings)
            
            # Memory cleanup
            del input_ids, attention_mask, outputs
            if i % 320 == 0:
                torch.cuda.empty_cache()
        
        return torch.cat(all_embeddings, dim=0)
    
    text_features = {}
    for name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
        if df is not None:
            texts = df['context_text'].tolist()
            text_features[name] = extract_text_embeddings(texts)
            print(f'  {name}: {text_features[name].shape}')
    
    # Save to disk
    torch.save(text_features, text_feat_path)
    print(f'✅ Text features saved to {text_feat_path}')
    
    # Free GPU memory
    del text_model, text_tokenizer
    torch.cuda.empty_cache()
    gc.collect()

In [ ]:
# === 4.2 Audio Feature Extraction (WavLM) ===
import tempfile
import struct
import wave

# Auto-detect WavLM model path
AUDIO_MODEL_PATH = None
for candidate in [
    '/kaggle/input/wavlm-base',
    '/kaggle/input/models/microsoft/wavlm/pytorch/base',
    '/kaggle/input/wavlm-base-plus',
]:
    if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, 'config.json')):
        AUDIO_MODEL_PATH = candidate
        break

if AUDIO_MODEL_PATH is None:
    configs = glob.glob('/kaggle/input/**/config.json', recursive=True)
    for c in configs:
        try:
            with open(c) as f:
                if 'wavlm' in f.read().lower():
                    AUDIO_MODEL_PATH = os.path.dirname(c)
                    break
        except Exception:
            pass

audio_feat_path = os.path.join(FEATURE_DIR, 'audio_features.pt')

if AUDIO_MODEL_PATH is None:
    print('⚠️ WavLM model not found. Audio features will be zero vectors.')
    print('   To enable: search Kaggle Models for "wavlm-base" and attach it.')
    # Create zero-vector fallback
    audio_features = {}
    for name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
        if df is not None:
            audio_features[name] = torch.zeros(len(df), AUDIO_DIM)
    torch.save(audio_features, audio_feat_path)
    print(f'✅ Zero-vector audio features saved (placeholder)')

elif os.path.isfile(audio_feat_path):
    print(f'✅ Audio features already extracted, loading from {audio_feat_path}')
    audio_features = torch.load(audio_feat_path)

else:
    print(f'Audio model: {AUDIO_MODEL_PATH}')
    print('Extracting audio features with WavLM...')
    
    audio_processor = AutoFeatureExtractor.from_pretrained(AUDIO_MODEL_PATH)
    audio_model = WavLMModel.from_pretrained(AUDIO_MODEL_PATH)
    audio_model = audio_model.to(DEVICE).float()
    if torch.cuda.device_count() > 1:
        audio_model = nn.DataParallel(audio_model)
    audio_model.eval()
    
    TARGET_SR = 16000  # WavLM expects 16kHz
    MAX_AUDIO_LEN = 5 * TARGET_SR  # Cap at 5 seconds to save memory
    
    def extract_audio_from_video(video_path):
        """Extract audio waveform from an MP4 file."""
        try:
            if HAS_TORCHAUDIO:
                waveform, sr = torchaudio.load(video_path)
                # Convert to mono if stereo
                if waveform.shape[0] > 1:
                    waveform = waveform.mean(dim=0, keepdim=True)
                # Resample to 16kHz
                if sr != TARGET_SR:
                    resampler = torchaudio.transforms.Resample(sr, TARGET_SR)
                    waveform = resampler(waveform)
                waveform = waveform.squeeze(0)
                # Truncate to max length
                if len(waveform) > MAX_AUDIO_LEN:
                    waveform = waveform[:MAX_AUDIO_LEN]
                return waveform.numpy()
            else:
                # Fallback: use ffmpeg to extract raw audio
                tmp_wav = tempfile.mktemp(suffix='.wav')
                subprocess.run(
                    ['ffmpeg', '-i', video_path, '-ar', str(TARGET_SR),
                     '-ac', '1', '-f', 'wav', '-y', tmp_wav],
                    capture_output=True, timeout=30
                )
                if os.path.isfile(tmp_wav):
                    with wave.open(tmp_wav, 'rb') as wf:
                        frames = wf.readframes(wf.getnframes())
                        audio = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
                    os.remove(tmp_wav)
                    if len(audio) > MAX_AUDIO_LEN:
                        audio = audio[:MAX_AUDIO_LEN]
                    return audio
                return None
        except Exception:
            return None
    
    def extract_audio_embedding(waveform_np):
        """Get WavLM embedding (mean-pooled) from a waveform."""
        if waveform_np is None or len(waveform_np) < 400:  # Too short
            return torch.zeros(AUDIO_DIM)
        
        inputs = audio_processor(
            waveform_np, sampling_rate=TARGET_SR,
            return_tensors='pt', padding=True
        )
        input_values = inputs['input_values'].to(DEVICE)
        
        with torch.no_grad():
            outputs = audio_model(input_values)
            # Mean-pool across time dimension
            embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0).cpu()
        
        del input_values, outputs
        return embedding
    
    audio_features = {}
    for split_name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
        if df is None:
            continue
        embeddings = []
        success = 0
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f'Audio ({split_name})'):
            vpath = row.get('video_path')
            if vpath and os.path.isfile(str(vpath)):
                waveform = extract_audio_from_video(str(vpath))
                emb = extract_audio_embedding(waveform)
                if emb.sum() != 0:
                    success += 1
            else:
                emb = torch.zeros(AUDIO_DIM)
            embeddings.append(emb)
            
            # Periodic GPU cleanup
            if idx % 200 == 0:
                torch.cuda.empty_cache()
        
        audio_features[split_name] = torch.stack(embeddings)
        print(f'  {split_name}: {audio_features[split_name].shape} ({success}/{len(df)} extracted)')
    
    torch.save(audio_features, audio_feat_path)
    print(f'✅ Audio features saved to {audio_feat_path}')
    
    del audio_model, audio_processor
    torch.cuda.empty_cache()
    gc.collect()


In [ ]:
# === 4.3 Video Feature Extraction (EfficientNet-B2 + Face Detection) ===

# Face detection setup (Haar Cascade — works offline on Kaggle)
print('Initializing MTCNN Face Detector...')
mtcnn = MTCNN(keep_all=True, device=DEVICE)
print('✅ MTCNN Face detector loaded')

# Image transforms for EfficientNet-B2 (native input: 260x260)
effnet_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

video_feat_path = os.path.join(FEATURE_DIR, 'video_features.pt')

if os.path.isfile(video_feat_path):
    print(f'✅ Video features already extracted, loading from {video_feat_path}')
    video_features = torch.load(video_feat_path)

else:
    print('Extracting video features with EfficientNet-B2...')
    
    # Load EfficientNet-B2 (remove final classification layer)
    try:
        effnet = models.efficientnet_b2(weights='IMAGENET1K_V1')
        print('  ✅ Loaded EfficientNet-B2 with ImageNet weights')
    except Exception:
        try:
            effnet = models.efficientnet_b2(pretrained=True)
            print('  ✅ Loaded EfficientNet-B2 with pretrained weights')
        except Exception:
            # Try loading from Kaggle input
            effnet_weights = glob.glob('/kaggle/input/**/efficientnet_b2*.pth', recursive=True)
            if not effnet_weights:
                effnet_weights = glob.glob('/kaggle/input/**/efficientnet*.pth', recursive=True)
            if effnet_weights:
                effnet = models.efficientnet_b2(weights=None)
                effnet.load_state_dict(torch.load(effnet_weights[0], map_location='cpu'))
                print(f'  ✅ Loaded EfficientNet-B2 weights from: {effnet_weights[0]}')
            else:
                print('  ⚠️ EfficientNet-B2 weights not found. Using random initialization.')
                print('     To fix: search Kaggle Datasets for "efficientnet-b2" and attach it.')
                effnet = models.efficientnet_b2(weights=None)
    
    # Remove the final classifier — keep the adaptive avg pool output (1408-d)
    effnet.classifier = nn.Identity()  # Output: [B, 1408]
    effnet = effnet.to(DEVICE).float()
    if torch.cuda.device_count() > 1:
        effnet = nn.DataParallel(effnet)
    effnet.eval()
    
    NUM_FRAMES = 5  # Sample 3 evenly-spaced frames per video
    
    def extract_faces_from_video(video_path, num_frames=NUM_FRAMES):
        """Extract face crops (or center crops) from a video."""
        if video_path is None or not os.path.isfile(str(video_path)):
            return None
        
        try:
            cap = cv2.VideoCapture(str(video_path))
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames <= 0:
                cap.release()
                return None
            
            # Calculate evenly-spaced frame indices
            if total_frames <= num_frames:
                indices = list(range(total_frames))
            else:
                indices = np.linspace(0, total_frames - 1, num_frames, dtype=int).tolist()
            
            face_images = []
            for idx in indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                ret, frame = cap.read()
                if not ret:
                    continue
                
                # Convert to RGB
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                
                # Detect faces using MTCNN
                boxes, probs = mtcnn.detect(frame_rgb)
                
                if boxes is not None and len(boxes) > 0:
                    # Take the face with the highest probability
                    best_idx = probs.argmax()
                    x1, y1, x2, y2 = boxes[best_idx]
                    
                    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
                    w, h = x2 - x1, y2 - y1
                    
                    # Add margin
                    margin = int(0.15 * max(w, h))
                    y1 = max(0, y1 - margin)
                    y2 = min(frame_rgb.shape[0], y2 + margin)
                    x1 = max(0, x1 - margin)
                    x2 = min(frame_rgb.shape[1], x2 + margin)
                    face_crop = frame_rgb[y1:y2, x1:x2]

                
                if face_crop.size > 0:
                    face_images.append(face_crop)
            
            cap.release()
            return face_images if face_images else None
        
        except Exception:
            return None
    
    def extract_video_embedding(face_images):
        """Get EfficientNet-B2 embedding from face crops (averaged across frames)."""
        if face_images is None or len(face_images) == 0:
            return torch.zeros(VIDEO_DIM)
        
        tensors = []
        for img in face_images:
            try:
                t = effnet_transform(img)
                tensors.append(t)
            except Exception:
                continue
        
        if not tensors:
            return torch.zeros(VIDEO_DIM)
        
        batch = torch.stack(tensors).to(DEVICE)
        with torch.no_grad():
            features = effnet(batch)  # [N, 1408]
            # Average across frames
            embedding = features.mean(dim=0).cpu()
        
        del batch, features
        return embedding
    
    video_features = {}
    for split_name, df in [('train', df_train), ('dev', df_dev), ('test', df_test)]:
        if df is None:
            continue
        embeddings = []
        success = 0
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f'Video ({split_name})'):
            vpath = row.get('video_path')
            faces = extract_faces_from_video(vpath)
            emb = extract_video_embedding(faces)
            if emb.sum() != 0:
                success += 1
            embeddings.append(emb)
            
            if idx % 100 == 0:
                torch.cuda.empty_cache()
        
        video_features[split_name] = torch.stack(embeddings)
        print(f'  {split_name}: {video_features[split_name].shape} ({success}/{len(df)} extracted)')
    
    torch.save(video_features, video_feat_path)
    print(f'✅ Video features saved to {video_feat_path}')
    
    del effnet
    torch.cuda.empty_cache()
    gc.collect()


In [ ]:
# === 4.4 Feature Extraction Summary ===

print('=' * 60)
print('FEATURE EXTRACTION SUMMARY')
print('=' * 60)

# Reload if needed
if 'text_features' not in dir():
    text_features = torch.load(os.path.join(FEATURE_DIR, 'text_features.pt'))
if 'audio_features' not in dir():
    audio_features = torch.load(os.path.join(FEATURE_DIR, 'audio_features.pt'))
if 'video_features' not in dir():
    video_features = torch.load(os.path.join(FEATURE_DIR, 'video_features.pt'))

for split in ['train', 'dev', 'test']:
    print(f'\n{split.upper()}:')
    if split in text_features:
        t = text_features[split]
        print(f'  Text  : {t.shape} | non-zero: {(t.abs().sum(dim=1) > 0).sum()}/{len(t)}')
    if split in audio_features:
        a = audio_features[split]
        print(f'  Audio : {a.shape} | non-zero: {(a.abs().sum(dim=1) > 0).sum()}/{len(a)}')
    if split in video_features:
        v = video_features[split]
        print(f'  Video : {v.shape} | non-zero: {(v.abs().sum(dim=1) > 0).sum()}/{len(v)}')

print('\n✅ All features ready for fusion training!')

---
## Phase 2: Fusion Model Training

Now that all features are saved to disk, we train a lightweight attention-based fusion model.  
This phase uses minimal GPU memory and trains in minutes.

In [ ]:
# === 5.1 Multimodal Dataset ===

class MultimodalDataset(Dataset):
    """Dataset that returns pre-extracted embeddings from all 3 modalities."""
    def __init__(self, text_embs, audio_embs, video_embs, labels):
        self.text = text_embs
        self.audio = audio_embs
        self.video = video_embs
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return {
            'text': self.text[idx],
            'audio': self.audio[idx],
            'video': self.video[idx],
            'label': self.labels[idx]
        }

# Create datasets
train_ds = MultimodalDataset(
    text_features['train'], audio_features['train'], video_features['train'], y_train
)
dev_ds = MultimodalDataset(
    text_features['dev'], audio_features['dev'], video_features['dev'], y_dev
)
test_ds = MultimodalDataset(
    text_features['test'], audio_features['test'], video_features['test'], y_test
)

BATCH_SIZE = 128  # Can be large since we're only loading small embedding vectors

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train: {len(train_ds)} | Dev: {len(dev_ds)} | Test: {len(test_ds)}')
print(f'Batch size: {BATCH_SIZE}')
print('✅ Multimodal dataloaders created!')

In [ ]:
# === 5.2 Attention-Based Fusion Model ===

class MultimodalFusionModel(nn.Module):
    """
    Attention-based fusion of Text, Audio, and Video embeddings.
    Each modality is projected to a common dimension, then fused
    via multi-head self-attention before classification.
    """
    def __init__(self, text_dim=TEXT_DIM, audio_dim=AUDIO_DIM, video_dim=VIDEO_DIM,
                 proj_dim=256, num_heads=4, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        
        # Project each modality to a common dimension
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.audio_proj = nn.Sequential(
            nn.Linear(audio_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.video_proj = nn.Sequential(
            nn.Linear(video_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        # Self-attention fusion
        self.attention = nn.MultiheadAttention(
            embed_dim=proj_dim, num_heads=num_heads,
            dropout=dropout, batch_first=True
        )
        self.attn_norm = nn.LayerNorm(proj_dim)
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, text_emb, audio_emb, video_emb):
        # Project to common dimension: each becomes [B, proj_dim]
        t = self.text_proj(text_emb).unsqueeze(1)    # [B, 1, proj_dim]
        a = self.audio_proj(audio_emb).unsqueeze(1)  # [B, 1, proj_dim]
        v = self.video_proj(video_emb).unsqueeze(1)   # [B, 1, proj_dim]
        
        # Stack as 3 "tokens" for self-attention
        tokens = torch.cat([t, a, v], dim=1)  # [B, 3, proj_dim]
        
        # Self-attention: each modality attends to the others
        fused, attn_weights = self.attention(tokens, tokens, tokens)
        fused = self.attn_norm(fused + tokens)  # Residual connection
        
        # Pool across modalities
        fused = fused.mean(dim=1)  # [B, proj_dim]
        
        return self.classifier(fused), attn_weights

# Initialize model
fusion_model = MultimodalFusionModel().to(DEVICE)

total_params = sum(p.numel() for p in fusion_model.parameters())
print(f'Fusion Model Parameters: {total_params:,}')
print(f'\nArchitecture:')
print(fusion_model)
print(f'\n✅ Fusion model created! (only {total_params/1e6:.2f}M params — trains in minutes)')

In [ ]:
# === 5.3 Focal Loss ===

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance.
    Down-weights easy examples and focuses on hard, misclassified samples."""
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

# Class weights for imbalanced MELD dataset
class_counts = np.bincount(y_train, minlength=NUM_CLASSES).astype(float)
class_weights = torch.tensor(1.0 / (class_counts + 1e-6), dtype=torch.float32).to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

criterion = FocalLoss(gamma=2.0, weight=class_weights)

print('Class distribution (train):')
for i, label in enumerate(EMOTION_LABELS):
    print(f'  {label:10s}: {int(class_counts[i]):5d} samples | weight: {class_weights[i]:.4f}')
print(f'\n✅ Focal Loss initialized (gamma=2.0)')

In [ ]:
# === 5.4 Training Loop ===

NUM_EPOCHS = 50  # Fusion model is tiny — can train many epochs quickly
LR = 1e-3
PATIENCE = 10    # Early stopping patience

optimizer = optim.AdamW(fusion_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

best_f1 = 0
patience_counter = 0
history = {'train_loss': [], 'dev_acc': [], 'dev_f1': [], 'dev_f1_macro': []}

print(f'Training fusion model for up to {NUM_EPOCHS} epochs (early stopping: patience={PATIENCE})\n')

for epoch in range(NUM_EPOCHS):
    # === Train ===
    fusion_model.train()
    total_loss = 0
    
    for batch in train_loader:
        text_emb  = batch['text'].to(DEVICE)
        audio_emb = batch['audio'].to(DEVICE)
        video_emb = batch['video'].to(DEVICE)
        labels    = batch['label'].to(DEVICE)
        
                # Modality Dropout: 10% chance to drop each modality during training
        # This forces the model to learn how to predict if the user only provides 1 or 2 modalities
        drop_t = random.random() < 0.1
        drop_a = random.random() < 0.1
        drop_v = random.random() < 0.1
        # Make sure we don't drop all 3 at once
        if drop_t and drop_a and drop_v: drop_t = False
        
        if drop_t: text_emb = torch.zeros_like(text_emb)
        if drop_a: audio_emb = torch.zeros_like(audio_emb)
        if drop_v: video_emb = torch.zeros_like(video_emb)
        
        optimizer.zero_grad()
        logits, _ = fusion_model(text_emb, audio_emb, video_emb)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    # === Validate ===
    fusion_model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in dev_loader:
            text_emb  = batch['text'].to(DEVICE)
            audio_emb = batch['audio'].to(DEVICE)
            video_emb = batch['video'].to(DEVICE)
            labels    = batch['label']
            
            logits, _ = fusion_model(text_emb, audio_emb, video_emb)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    
    dev_acc = accuracy_score(all_labels, all_preds)
    dev_f1w = f1_score(all_labels, all_preds, average='weighted')
    dev_f1m = f1_score(all_labels, all_preds, average='macro')
    
    history['train_loss'].append(avg_loss)
    history['dev_acc'].append(dev_acc)
    history['dev_f1'].append(dev_f1w)
    history['dev_f1_macro'].append(dev_f1m)
    
    scheduler.step(dev_f1w)
    
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | '
          f'Dev Acc: {dev_acc:.4f} | F1w: {dev_f1w:.4f} | F1m: {dev_f1m:.4f}', end='')
    
    if dev_f1w > best_f1:
        best_f1 = dev_f1w
        patience_counter = 0
        state_dict_to_save = fusion_model.module.state_dict() if isinstance(fusion_model, nn.DataParallel) else fusion_model.state_dict()
        torch.save(state_dict_to_save, '/kaggle/working/best_fusion_model.pt')
        print(f'  🏆 Best! Saved.')
    else:
        patience_counter += 1
        print(f'  (patience: {patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'\n⏹️ Early stopping triggered at epoch {epoch+1}')
            break

print(f'\n✅ Training complete! Best F1w: {best_f1:.4f}')


In [ ]:
# === 5.5 Training Curves ===

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], 'b-o', markersize=3)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Focal Loss')

axes[1].plot(history['dev_acc'], 'g-o', markersize=3, label='Accuracy')
axes[1].plot(history['dev_f1'], 'r-o', markersize=3, label='F1 Weighted')
axes[1].set_title('Dev Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(history['dev_f1_macro'], 'm-o', markersize=3, label='F1 Macro')
axes[2].set_title('Dev F1 Macro (Minority Class Indicator)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.suptitle('Multimodal Fusion — Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 6: Evaluation

In [ ]:
# === 6.1 Load Best Model & Evaluate on Test Set ===

fusion_model.load_state_dict(torch.load('/kaggle/working/best_fusion_model.pt'))
fusion_model.eval()

test_preds, test_labels = [], []
all_attn_weights = []

with torch.no_grad():
    for batch in test_loader:
        text_emb  = batch['text'].to(DEVICE)
        audio_emb = batch['audio'].to(DEVICE)
        video_emb = batch['video'].to(DEVICE)
        labels    = batch['label']
        
        logits, attn_w = fusion_model(text_emb, audio_emb, video_emb)
        preds = logits.argmax(dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_labels.extend(labels.numpy())
        all_attn_weights.append(attn_w.cpu())

test_acc = accuracy_score(test_labels, test_preds)
test_f1w = f1_score(test_labels, test_preds, average='weighted')
test_f1m = f1_score(test_labels, test_preds, average='macro')

print('╔══════════════════════════════════════════════════════════╗')
print('║        MULTIMODAL FUSION — TEST SET RESULTS             ║')
print('╚══════════════════════════════════════════════════════════╝')
print(f'  Accuracy    : {test_acc:.4f}')
print(f'  F1 Weighted : {test_f1w:.4f}')
print(f'  F1 Macro    : {test_f1m:.4f}')

In [ ]:
# === 6.2 Classification Report ===
print('\n=== Classification Report (Test Set) ===\n')
print(classification_report(test_labels, test_preds, target_names=le.classes_))

In [ ]:
# === 6.3 Confusion Matrix ===
cm = confusion_matrix(test_labels, test_preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Multimodal Fusion (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# === 6.4 Modality Attention Analysis ===
# How much does the model rely on each modality?

attn = torch.cat(all_attn_weights, dim=0)  # [N, num_heads, 3, 3]
attn_avg = attn.mean(dim=(0, 1))  # Average across all samples and heads: [3, 3]

# The diagonal tells us how much each modality "self-attends" vs cross-attends
modality_names = ['Text', 'Audio', 'Video']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Attention heatmap
sns.heatmap(attn_avg.numpy(), annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=modality_names, yticklabels=modality_names, ax=axes[0])
axes[0].set_title('Cross-Modal Attention Weights (Avg)')
axes[0].set_xlabel('Key (Attended To)')
axes[0].set_ylabel('Query (Attending From)')

# Per-modality importance (sum of attention received from all modalities)
importance = attn_avg.sum(dim=0).numpy()
importance = importance / importance.sum() * 100
colors = ['#3498DB', '#E67E22', '#2ECC71']
axes[1].bar(modality_names, importance, color=colors)
axes[1].set_title('Modality Importance (%)')
axes[1].set_ylabel('Relative Importance (%)')
for i, v in enumerate(importance):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontweight='bold')

plt.suptitle('Which Modality Does the Model Rely On?', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 7: Inference & Submission

In [ ]:
# === 7.1 Flexible Inference Demo ===

def predict_emotion(text=None, video_path=None, prev_text=""):
    """
    Flexible inference: Provide text only, video only, or both!
    The model handles missing inputs by passing zero-vectors to the unused brains.
    """
    if not text and not video_path:
        print("❌ Error: You must provide either text or a video_path (or both).")
        return None
        
    fusion_model.eval()
    
    # --- Text Brain ---
    if text:
        cleaned = clean_text(text)
        cleaned_prev = clean_text(prev_text) if prev_text else ""
        
        text_tok = AutoTokenizer.from_pretrained(TEXT_MODEL_PATH)
        text_mdl = AutoModel.from_pretrained(TEXT_MODEL_PATH).to(DEVICE).float().eval()
        
        inputs = text_tok(cleaned, cleaned_prev, max_length=128, padding='max_length', truncation=True, return_tensors='pt')
        with torch.no_grad():
            text_out = text_mdl(inputs['input_ids'].to(DEVICE), inputs['attention_mask'].to(DEVICE))
            text_emb = text_out.last_hidden_state[:, 0, :].cpu()
        del text_mdl, text_tok
        torch.cuda.empty_cache()
    else:
        print("⚠️ No text provided. Text Brain is disabled.")
        text_emb = torch.zeros(1, TEXT_DIM)
    
    # --- Audio & Video Brains ---
    if video_path and os.path.isfile(video_path):
        # Audio
        if AUDIO_MODEL_PATH:
            a_proc = AutoFeatureExtractor.from_pretrained(AUDIO_MODEL_PATH)
            a_mdl = WavLMModel.from_pretrained(AUDIO_MODEL_PATH).to(DEVICE).float().eval()
            waveform = extract_audio_from_video(video_path)
            if waveform is not None and len(waveform) >= 400:
                a_input = a_proc(waveform, sampling_rate=16000, return_tensors='pt', padding=True)
                with torch.no_grad():
                    a_out = a_mdl(a_input['input_values'].to(DEVICE))
                    audio_emb = a_out.last_hidden_state.mean(dim=1).cpu()
            else:
                audio_emb = torch.zeros(1, AUDIO_DIM)
            del a_mdl, a_proc
            torch.cuda.empty_cache()
        else:
            audio_emb = torch.zeros(1, AUDIO_DIM)
            
        # Video
        faces = extract_faces_from_video(video_path)
        if faces:
            tensors = [effnet_transform(img) for img in faces]
            batch = torch.stack(tensors).to(DEVICE)
            try:
                effnet_inf = models.efficientnet_b2(weights='IMAGENET1K_V1')
            except Exception:
                effnet_inf = models.efficientnet_b2(weights=None)
            effnet_inf.classifier = nn.Identity()
            effnet_inf = effnet_inf.to(DEVICE).float().eval()
            with torch.no_grad():
                v_feat = effnet_inf(batch).mean(dim=0, keepdim=True).cpu()
            video_emb = v_feat
            del effnet_inf
            torch.cuda.empty_cache()
        else:
            video_emb = torch.zeros(1, VIDEO_DIM)
    else:
        if video_path:
            print(f"⚠️ Video not found at {video_path}.")
        print("⚠️ Audio and Video Brains are disabled.")
        audio_emb = torch.zeros(1, AUDIO_DIM)
        video_emb = torch.zeros(1, VIDEO_DIM)
    
    # --- Fusion ---
    with torch.no_grad():
        logits, attn_w = fusion_model(
            text_emb.to(DEVICE), audio_emb.to(DEVICE), video_emb.to(DEVICE)
        )
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
    
    pred_idx = probs.argmax()
    emotion = le.inverse_transform([pred_idx])[0]
    
    # Display results
    emoji = {'anger':'😠','disgust':'🤢','fear':'😨','joy':'😊',
             'neutral':'😐','sadness':'😢','surprise':'😲'}.get(emotion, '❓')
    
    print(f'\nPredicted Emotion: {emoji} {emotion} ({probs.max():.2%})')
    print(f'\nProbabilities:')
    for i, label in enumerate(EMOTION_LABELS):
        bar = '█' * int(probs[i] * 40)
        print(f'  {label:10s} {probs[i]:.4f} {bar}')
    
    return emotion, probs

# Demo 1: Full Multimodal
demo_row = df_test[df_test['video_path'].notna()].iloc[0] if df_test['video_path'].notna().any() else None
if demo_row is not None:
    print('=== DEMO 1: TEXT + AUDIO + VIDEO ===')
    predict_emotion(text=demo_row['Utterance'], video_path=demo_row['video_path'])

# Demo 2: Text Only
print('\n=== DEMO 2: TEXT ONLY (Like a Chatbot) ===')
predict_emotion(text="I am absolutely furious right now!", video_path=None)


In [ ]:
# === 7.2 Generate submission.csv ===

fusion_model.load_state_dict(torch.load('/kaggle/working/best_fusion_model.pt'))
fusion_model.eval()

submission_preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Generating predictions'):
        text_emb  = batch['text'].to(DEVICE)
        audio_emb = batch['audio'].to(DEVICE)
        video_emb = batch['video'].to(DEVICE)
        
        logits, _ = fusion_model(text_emb, audio_emb, video_emb)
        preds = logits.argmax(dim=1).cpu().numpy()
        submission_preds.extend(preds)

# Map predictions back to emotion labels
predicted_emotions = le.inverse_transform(submission_preds)

# Create submission dataframe
submission = pd.DataFrame({
    'id': range(len(predicted_emotions)),
    'Emotion': predicted_emotions
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print(f'\n✅ Submission saved: /kaggle/working/submission.csv')
print(f'Shape: {submission.shape}')
print(f'\nPrediction distribution:')
print(submission['Emotion'].value_counts().to_string())

In [ ]:
# === 7.3 Final Summary ===

print('╔══════════════════════════════════════════════════════════╗')
print('║            MULTIMODAL EMOTION RECOGNITION               ║')
print('║                    FINAL SUMMARY                        ║')
print('╚══════════════════════════════════════════════════════════╝')
print(f'\n  Architecture:')
print(f'    Text  : DeBERTa-v3-base  ({TEXT_DIM}-d embeddings)')
print(f'    Audio : WavLM-base       ({AUDIO_DIM}-d embeddings)')
print(f'    Video : EfficientNet-B2        ({VIDEO_DIM}-d embeddings)')
print(f'    Fusion: Multi-Head Self-Attention (4 heads, 256-d)')
print(f'\n  Test Results:')
print(f'    Accuracy    : {test_acc:.4f}')
print(f'    F1 Weighted : {test_f1w:.4f}')
print(f'    F1 Macro    : {test_f1m:.4f}')
print(f'\n  Files:')
print(f'    Features    : {FEATURE_DIR}/')
print(f'    Best Model  : /kaggle/working/best_fusion_model.pt')
print(f'    Submission  : /kaggle/working/submission.csv')
print(f'\n🏆 Done!')